# Фильтрация после аннотации — PRJEB30386

Фильтруются **те же объединённые pRESTO FASTQ**, которые были входом IgBLAST: `results/PRJEB30386/merged/fastq/*_assemble-pass.fastq.gz`, синхронно с `annotation/igblast/*.airr.tsv`. Обрезка адаптеров и праймеров на этом этапе не выполняется.

Критерии соответствуют контролю качества после аннотации: интервал V–J ≥300 nt; начало V присутствует (`v_germline_start == 1`); полнота J-конца по AIRR `complete_vdj=True`; ожидаемый локус для конкретной библиотеки; продуктивность; отсутствие стоп-кодонов; V/J identity и V/J IgBLAST E-value.

По текущему отчёту минимальный P01 для V identity среди четырёх библиотек = **83.391%** (IgG), для J identity = **88.372%**. Поэтому используются консервативные пороговые значения **V ≥80%** и **J ≥85%**, расположенные ниже наблюдаемого 1%-го хвоста. Для V/J support применяется **E-value ≤1e-5**; по текущему отчёту этот порог сохраняет ≥99.27% V и ≥98.94% J назначений. Это специфичной для набора данных QC, а не универсальные биологические пороги.


In [ ]:
import csv, gzip, json, shutil
from collections import Counter
from pathlib import Path

DATASET = "PRJEB30386"
LOCAL_REPO = Path("/Users/epishkin/workspace/bcr-assembler")
VOLUME_ROOT = Path("/data/user/epishkin")

MIN_VJ_LENGTH = 300
MIN_V_IDENTITY = 80.0   # В AIRR IgBLAST v_identity задана в процентах, а не долях.
MIN_J_IDENTITY = 85.0   # В AIRR IgBLAST j_identity задана в процентах, а не долях.
MAX_V_SUPPORT = 1e-5    # Для IgBLAST меньшее E-value соответствует более надёжному совпадению.
MAX_J_SUPPORT = 1e-5
EXPECTED_LOCUS = {
    "ERR3004229": "IGH",  # IgM
    "ERR3004230": "IGH",  # IgG
    "ERR3004231": "IGK",  # IgK
    "ERR3004232": "IGL",  # IgL
}
FORCE = False

def resolve_results_root():
    for root in (VOLUME_ROOT / "results", LOCAL_REPO / "results"):
        if (root / DATASET / "merged" / "fastq").is_dir():
            return root
    raise FileNotFoundError("PRJEB30386 merged/fastq not found")

RESULTS_ROOT = resolve_results_root()
DATASET_DIR = RESULTS_ROOT / DATASET
MERGED_DIR = DATASET_DIR / "merged" / "fastq"
ANNOT_DIR = DATASET_DIR / "annotation" / "igblast"
FINAL_DIR = DATASET_DIR / "post_annotation_filtered"
STAGING_DIR = DATASET_DIR / ".post_annotation_filtered.staging"
SAMPLES = sorted(p.name.removesuffix("_assemble-pass.fastq.gz") for p in MERGED_DIR.glob("*_assemble-pass.fastq.gz"))
assert SAMPLES == ["ERR3004229", "ERR3004230", "ERR3004231", "ERR3004232"], SAMPLES
print(MERGED_DIR, ANNOT_DIR, FINAL_DIR, sep="\n")


In [ ]:
TRUE = {"true", "t", "1", "yes", "y"}
FALSE = {"false", "f", "0", "no", "n"}

def as_bool(v):
    x = str(v).strip().lower()
    return True if x in TRUE else False if x in FALSE else None

def as_float(v):
    try:
        return float(v)
    except (TypeError, ValueError):
        return None

def as_int(v):
    try:
        return int(float(v))
    except (TypeError, ValueError):
        return None

REQUIRED = {
    "sequence_id", "sequence", "locus", "productive", "stop_codon", "complete_vdj",
    "v_sequence_start", "j_sequence_end", "v_germline_start",
    "v_identity", "j_identity", "v_support", "j_support",
}

def validate_schema(path):
    with open(path, newline="") as fh:
        fields = csv.DictReader(fh, delimiter="\t").fieldnames or []
    missing = sorted(REQUIRED - set(fields))
    if missing:
        raise ValueError(f"{path}: missing AIRR fields {missing}")

for s in SAMPLES:
    validate_schema(ANNOT_DIR / f"{s}.airr.tsv")
print("AIRR schemas OK")


In [ ]:
def evaluate(row, expected_locus):
    reasons = []

    # 1) Интервал V–J в аннотированном запросе применим к тяжёлым и лёгким цепям.
    v_start = as_int(row.get("v_sequence_start"))
    j_end = as_int(row.get("j_sequence_end"))
    if v_start is None or j_end is None:
        reasons.append("missing_vj_coordinates")
    elif j_end - v_start + 1 < MIN_VJ_LENGTH:
        reasons.append("vj_span_lt_300")

    # 2) Требуется наличие 5′-начала germline-референса V.
    if as_int(row.get("v_germline_start")) != 1:
        reasons.append("v_gene_5prime_incomplete")

    # 3) AIRR complete_vdj означает, что выравнивание включает первый кодирующий
    #    кодон V и последний полный кодон J. Это критерий полноты J-конца;
    #    равенство j_sequence_end == len(sequence) не требуется, поскольку
    #    объединённый рид может продолжаться в константную область.
    if as_bool(row.get("complete_vdj")) is not True:
        reasons.append("j_3prime_or_vdj_incomplete")

    # 4) Для библиотек задана цепь или изотип; назначения другого локуса исключаются.
    if str(row.get("locus", "")).strip() != expected_locus:
        reasons.append("unexpected_locus")

    # 5) Проверка продуктивности и стоп-кодонов.
    if as_bool(row.get("productive")) is not True:
        reasons.append("nonproductive")
    if as_bool(row.get("stop_codon")) is not False:
        reasons.append("stop_codon_or_missing")

    # 6) Проверка уверенности аннотации; identity в IgBLAST AIRR задана в процентах.
    vi = as_float(row.get("v_identity"))
    ji = as_float(row.get("j_identity"))
    vs = as_float(row.get("v_support"))
    js = as_float(row.get("j_support"))

    if vi is None:
        reasons.append("missing_v_identity")
    elif vi < MIN_V_IDENTITY:
        reasons.append("low_v_identity")

    if ji is None:
        reasons.append("missing_j_identity")
    elif ji < MIN_J_IDENTITY:
        reasons.append("low_j_identity")

    if vs is None:
        reasons.append("missing_v_support")
    elif vs > MAX_V_SUPPORT:
        reasons.append("poor_v_support")

    if js is None:
        reasons.append("missing_j_support")
    elif js > MAX_J_SUPPORT:
        reasons.append("poor_j_support")

    return reasons

print({
    "MIN_VJ_LENGTH": MIN_VJ_LENGTH,
    "MIN_V_IDENTITY": MIN_V_IDENTITY,
    "MIN_J_IDENTITY": MIN_J_IDENTITY,
    "MAX_V_SUPPORT": MAX_V_SUPPORT,
    "MAX_J_SUPPORT": MAX_J_SUPPORT,
    "EXPECTED_LOCUS": EXPECTED_LOCUS,
})


In [ ]:
def fastq_records(path):
    with gzip.open(path, "rt") as fh:
        while True:
            h = fh.readline()
            if not h:
                break
            seq, plus, qual = fh.readline(), fh.readline(), fh.readline()
            if not qual or not h.startswith("@") or not plus.startswith("+"):
                raise ValueError(f"Malformed FASTQ: {path}")
            yield h, seq, plus, qual

def fastq_id(header):
    return header[1:].strip().split(None, 1)[0]

def filter_sample(sample, out):
    fq = MERGED_DIR / f"{sample}_assemble-pass.fastq.gz"
    airr = ANNOT_DIR / f"{sample}.airr.tsv"
    pfq = out / "fastq" / f"{sample}_filtered.fastq.gz"
    ptsv = out / "airr_pass" / f"{sample}.airr.tsv"
    rtsv = out / "airr_reject" / f"{sample}.airr.tsv"

    counts = Counter()
    reason_counts = Counter()
    expected_locus = EXPECTED_LOCUS[sample]

    with open(airr, newline="") as af, \
         gzip.open(pfq, "wt") as ofq, \
         open(ptsv, "w", newline="") as op, \
         open(rtsv, "w", newline="") as orej:

        reader = csv.DictReader(af, delimiter="\t")
        fields = reader.fieldnames or []
        pass_writer = csv.DictWriter(op, fieldnames=fields, delimiter="\t", lineterminator="\n")
        reject_writer = csv.DictWriter(orej, fieldnames=fields + ["filter_reasons"], delimiter="\t", lineterminator="\n")
        pass_writer.writeheader()
        reject_writer.writeheader()
        fqit = fastq_records(fq)

        for i, row in enumerate(reader, 1):
            rec = next(fqit, None)
            if rec is None:
                raise ValueError(f"FASTQ ended early: {sample} row {i}")
            if fastq_id(rec[0]) != row["sequence_id"]:
                raise ValueError(f"ID/order mismatch {sample} row {i}: FASTQ={fastq_id(rec[0])!r}, AIRR={row['sequence_id']!r}")

            reasons = evaluate(row, expected_locus)
            counts["input"] += 1
            if reasons:
                counts["rejected"] += 1
                reason_counts.update(reasons)
                rr = dict(row)
                rr["filter_reasons"] = ";".join(reasons)
                reject_writer.writerow(rr)
            else:
                counts["passed"] += 1
                pass_writer.writerow(row)
                ofq.writelines(rec)

        if next(fqit, None) is not None:
            raise ValueError(f"FASTQ has extra records: {sample}")

    return {
        "input": counts["input"],
        "passed": counts["passed"],
        "rejected": counts["rejected"],
        "pass_pct": 100.0 * counts["passed"] / counts["input"] if counts["input"] else 0.0,
        "reasons": dict(reason_counts),
    }


In [ ]:
def _promote(staging, final):
    staging, final = Path(staging), Path(final)
    previous = final.parent / f".{final.name}.previous"

    if previous.exists():
        shutil.rmtree(previous)

    if final.exists():
        final.rename(previous)

    try:
        staging.rename(final)
    except Exception:
        if previous.exists() and not final.exists():
            previous.rename(final)
        raise

    if previous.exists():
        shutil.rmtree(previous)


def run_filter(force=FORCE):
    # Наличие готового результата проверяется до обработки примерно 3,8 млн записей.
    if FINAL_DIR.exists() and not force:
        raise FileExistsError(
            f"Output already exists: {FINAL_DIR}; "
            "set FORCE=True only if you intentionally want to replace it"
        )

    # Непустой промежуточный каталог считается признаком незавершённого результата.
    if STAGING_DIR.exists():
        if not force:
            raise FileExistsError(
                f"Staging output exists: {STAGING_DIR}; "
                "inspect/remove it or set FORCE=True to rebuild"
            )
        shutil.rmtree(STAGING_DIR)

    for sub in ("fastq", "airr_pass", "airr_reject"):
        (STAGING_DIR / sub).mkdir(parents=True, exist_ok=True)

    summary = {}
    for sample in SAMPLES:
        print(f"[filter] {sample}", flush=True)
        summary[sample] = filter_sample(sample, STAGING_DIR)
        s = summary[sample]
        print(
            f"  input={s['input']:,} passed={s['passed']:,} "
            f"({s['pass_pct']:.2f}%) rejected={s['rejected']:,}",
            flush=True,
        )

    qc = {
        "dataset": DATASET,
        "input": "merged/fastq/*_assemble-pass.fastq.gz (same records used for IgBLAST)",
        "filters": {
            "min_vj_span_nt": MIN_VJ_LENGTH,
            "require_v_germline_start_eq_1": True,
            "require_complete_vdj_for_j_3prime_end": True,
            "require_expected_locus": EXPECTED_LOCUS,
            "require_productive": True,
            "reject_stop_codon": True,
            "min_v_identity_percent": MIN_V_IDENTITY,
            "min_j_identity_percent": MIN_J_IDENTITY,
            "max_v_support_evalue": MAX_V_SUPPORT,
            "max_j_support_evalue": MAX_J_SUPPORT,
        },
        "cutoff_basis": {
            "v_identity": "80% is below the minimum observed library P01 (83.391%, IgG)",
            "j_identity": "85% is below the minimum observed library P01 (88.372%, IgG)",
            "support": "1e-5 retains >=99.27% of V and >=98.94% of J assignments in the current annotation report",
        },
        "samples": summary,
    }
    (STAGING_DIR / "filter_summary.json").write_text(json.dumps(qc, indent=2) + "\n")

    # Атомарная публикация выполняется только после того, как
    # все образцы и сводные файлы успешно записаны. При
    # FORCE=True существующий результат временно сохраняется для отката
    # до успешного переименования.
    _promote(STAGING_DIR, FINAL_DIR)

    print("DONE", FINAL_DIR)
    return qc


## Параметры и проверка

1. Отчёт формируется командой `python scripts/airr_igbrowser_report.py all`.
2. Перед фильтрацией проверяются распределения V identity, J identity, `-log10(V E-value)`, `-log10(J E-value)` и таблица предварительного просмотра.
3. Консервативные пороги для этого набора данных: `80%`, `85%`, `1e-5`, `1e-5`.
4. При `FORCE=False` существующий `post_annotation_filtered/` не изменяется. `FORCE=True` разрешает атомарную замену результата после полной проверки промежуточной стадии.


In [ ]:
qc = run_filter(force=FORCE)
